<a href="https://colab.research.google.com/github/withfablue/PyTorch/blob/main/Chapter_7_Transformer_architecture.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transformer Encoder-Decoder

In [3]:
# 1. Scaled dot product attention
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

def scaled_dot_product_attention(Q, K, V, mask=None):
   # Q, K, V shape: (batch, heads, seq_len, head_dim)
   # 점수 행렬: (batch, heads, seq_len_q, seq_len_k)
   scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(Q.size(-1))

   if mask is not None:
    scores = scores.masked_fill(mask, float('-inf'))

   attn = F.softmax(scores, dim=-1)  # (batch, heads, seq_len_q, seq_len_k)
   out = torch.matmul(attn, V)  # (batch, heads, seq_len_q, head_dim)
   return out, attn

In [2]:
# scaled dot product attention
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

def scaled_dot_product_attention(Q, K, V, mask=None):
  # Q, K, V -> b, h, t, hdim
  scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(Q.size(-1)) # b, h, t, t

  if mask is not None:
    scores = scores.masked_fill(mask, float('-inf'))

  attn = F.softmax(scores, dim=-1)
  out = torch.matmul(attn, V)
  return out, attn

In [4]:
# 2. Multihead attention

class MultiHeadAttention(nn.Module):
  def __init__(self, embed_dim, heads):
    super().__init__()
    assert embed_dim % heads == 0
    self.head_dim = embed_dim // heads
    self.heads = heads

    self.WQ = nn.Linear(embed_dim, embed_dim)
    self.WK = nn.Linear(embed_dim, embed_dim)
    self.WV = nn.Linear(embed_dim, embed_dim)
    self.W0 = nn.Linear(embed_dim, embed_dim)

  def forward(self, x, mask=None, kv=None):
    """
    x: 쿼리를 구성하는 입력, (batch, seq_len_q, embed_dim)
    kv: 키,값을 구성하는 입력. cross-attention 시 인코더 출력이 들어옴.
        None이면 셀프 어텐션으로 동작
    """
    K_input = x if kv is None else kv

    batch, seq_len_q, embed_dim = x.shape
    seq_len_k = K_input.size(1)

    # 선형 변환 후 (batch, heads, seq_len, head_dim) 형태로 변환
    Q = self.WQ(x).view(batch, seq_len_q, self.heads, self.head_dim).transpose(1, 2)
    K = self.WK(K_input).view(batch, seq_len_k, self.heads, self.head_dim).transpose(1, 2)
    V = self.WV(K_input).view(batch, seq_len_k, self.heads, self.head_dim).transpose(1, 2)

    out, attn = scaled_dot_product_attention(Q, K, V, mask)
    # 다시 (batch, seq_len, embed_dim) 으로 되돌린다
    out = out.transpose(1, 2).contiguous().view(batch, seq_len_q, embed_dim)
    return self.W0(out)

In [ ]:
# Multihead Attention
class MultiHeadAttention(nn.Module):
  def __init__(self, embed_dim, heads):
    super().__init__()
    assert embed_dim % heads == 0
    self.head_dim = embed_dim // heads
    self.heads = heads

    self.WQ = nn.Linear(embed_dim, embed_dim)
    self.WK = nn.Linear(embed_dim, embed_dim)
    self.WV = nn.Linear(embed_dim, embed_dim)
    self.W0 = nn.Linear(embed_dim, embed_dim)

  def forward(self, x, mask=None, kv=None):
    K_input = x if kv is None else kv

    batch, seq_len_q, embed_size = x.shape
    seq_len_k = K_input.size(1)

    Q = self.WQ(x).view(batch, seq_len_q, self.heads, self.head_dim).transpose(1, 2)
    K = self.WK(K_input).view(batch, seq_len_k, self.heads, self.head_dim).transpose(1, 2)
    V = self.WV(K_input).view(batch, seq_len_k, self.heads, self.head_dim).transpose(1, 2)

    out, attn = scaled_dot_product_attention(Q, K, V, mask)

    out = out.transpose(1, 2).contiguous().view(batch, seq_len_q, embed_dim)

    return self.W0(out)


In [5]:
# 3. Encoder
class EncoderLayer(nn.Module):
  def __init__(self, embed_dim, heads, ff_dim):
    super().__init__()
    self.attn = MultiHeadAttention(embed_dim, heads)
    self.norm1 = nn.LayerNorm(embed_dim)

    self.ff = nn.Sequential(
        nn.Linear(embed_dim, ff_dim),
        nn.ReLU(),
        nn.Linear(ff_dim, embed_dim)
    )
    self.norm2 = nn.LayerNorm(embed_dim)

  def forward(self, x):
    # x shape: (batch, seq_len, embed_dim)

    # Self-Attention + LayerNorm + Residual connection (Post LN)
    h = self.attn(x)
    x = self.norm1(x + h)

    # FFN + residual connection + norm
    h2 = self.ff(x)
    x = self.norm2(x + h2)
    return x

In [6]:
# 4. Decoder
class DecoderLayer(nn.Module):
  def __init__(self, embed_dim, heads, ff_dim):
    super().__init__()
    self.self_attn = MultiHeadAttention(embed_dim, heads)
    self.norm1 = nn.LayerNorm(embed_dim)

    self.cross_attn = MultiHeadAttention(embed_dim, heads)
    self.norm2 = nn.LayerNorm(embed_dim)

    self.ff = nn.Sequential(
        nn.Linear(embed_dim, ff_dim),
        nn.ReLU(),
        nn.Linear(ff_dim, embed_dim)
    )
    self.norm3 = nn.LayerNorm(embed_dim)

  def forward(self, x, enc_out, mask=None):
    # x: 디코더 입력(부분 생성 시퀀스), enc_out: 인코더 출력

    # Masked self-attention
    h = self.self_attn(x, mask=mask)
    x = self.norm1(x + h)

    # Encoder-Decoder cross attention
    h2 = self.cross_attn(x, kv=enc_out)
    x = self.norm2(x + h2)

    # Feedforward network
    h3 = self.ff(x)
    out = self.norm3(x + h3)

    return out


"""
class DecoderLayer(nn.Module):
  def __init__(self, embed_dim, heads, ff_dim):
    super().__init__()
    self.self_attn = MultiHeadAttention(embed_dim, heads)
    self.norm1 = nn.LayerNorm(embed_dim)

    self.cross_attn = MultiHeadAttention(embed_dim, heads)
    self.norm2 = nn.LayerNorm(embed_dim)

    self.ff = nn.Sequential(nn.Linear(embed_dim, ff_dim), nn.ReLU(), nn.Linear(ff_dim, embed_dim))
    self.norm3 = nn.LayerNorm(embed_dim)

  def forward(self, x, enc_out, mask=None):
    h = self.self_attn(x, mask=mask)
    x = self.norm1(x + h)

    h2 = self.cross_attn(x, kv=enc_out)
    x = self.norm2(x + h2)

    h3 = self.ff(x)
    x = self.norm3(x + h3)
    return x

  """

'\nclass DecoderLayer(nn.Module):\n  def __init__(self, embed_dim, heads, ff_dim):\n    super().__init__()\n    self.self_attn = MultiHeadAttention(embed_dim, heads)\n    self.norm1 = nn.LayerNorm(embed_dim)\n\n    self.cross_attn = MultiHeadAttention(embed_dim, heads)\n    self.norm2 = nn.LayerNorm(embed_dim)\n\n    self.ff = nn.Sequential(nn.Linear(embed_dim, ff_dim), nn.ReLU(), nn.Linear(ff_dim, embed_dim))\n    self.norm3 = nn.LayerNorm(embed_dim)\n\n  def forward(self, x, enc_out, mask=None):\n    h = self.self_attn(x, mask=mask)\n    x = self.norm1(x + h)\n\n    h2 = self.cross_attn(x, kv=enc_out)\n    x = self.norm2(x + h2)\n\n    h3 = self.ff(x)\n    x = self.norm3(x + h3)\n    return x\n\n  '

In [10]:
# 5. Transformer encoder-decoder model

class Transformer(nn.Module):
  def __init__(self, vocab_size, embed_dim, heads, ff_dim, depth, max_len=128):
    super().__init__()
    # Token embedding
    self.tok_embed = nn.Embedding(vocab_size, embed_dim)
    # Positional Encoding
    self.pos_embed = nn.Embedding(max_len, embed_dim)

    # Encoder, Decoder layers
    self.enc_layers = nn.ModuleList([
        EncoderLayer(embed_dim, heads, ff_dim) for _ in range(depth)
    ])
    self.dec_layers = nn.ModuleList([
        DecoderLayer(embed_dim, heads, ff_dim) for _ in range(depth)
    ])

    # Final output layer (Embedding -> Vocab distribution)
    self.fc_out = nn.Linear(embed_dim, vocab_size)

  def forward(self, src, tgt, tgt_mask=None):
     # src, tgt shape: (batch, seq_len)
     batch, src_len = src.shape
     _, tgt_len = tgt.shape
     device = src.device

     # 위치 인덱스 생성: 0, 1, ..., seq_len-1
     src_positions = torch.arange(src_len, device=device).unsqueeze(0)
     # (1, src_len)
     tgt_positions = torch.arange(tgt_len, device=device).unsqueeze(0)
     # (1, tgt_len)

     # Token embedding + Positional embedding
     src_emb = self.tok_embed(src) + self.pos_embed(src_positions)
     # batch, src_len, embed_dim
     tgt_emb = self.tok_embed(tgt) + self.pos_embed(tgt_positions)
     # batch, tgt_len, embed_dim

     # Encoder
     enc = src_emb
     for layer in self.enc_layers:
      enc = layer(enc)

     # Decoder
     dec = tgt_emb
     for layer in self.dec_layers:
      dec = layer(dec, enc, mask=tgt_mask)

     # Final output layer
     return self.fc_out(dec)  # (batch, tgt_len, vocab_size)


"""
class Transformer(nn.Module):
  def __init__(self, vocab_size, embed_dim, heads, ff_dim, depth, max_len=128):
    super().__init__()

    self.tok_embed = nn.Embedding(vocab_size, embed_dim)
    self.pos_embed = nn.Embedding(max_len, embed_dim)

    self.enc_layers = nn.ModuleList([
        EncoderLayer(embed_dim, heads, ff_dim) for _ in range(depth)
    ])

    self.dec_layers = nn.ModuleList([
        DecoderLayer(embed_dim, heads, ff_dim) for _ in range(depth)
    ])

    self.fc_out = nn.Linear(embed_dim, vocab_size)

  def forward(self, src, tgt, tgt_mask=None):
    batch, src_len = src.shape
    _, tgt_len = tgt.shape
    device = src.device

    src_positions = torch.arange(src_len, device=device).unsqueeze(0)
    tgt_positions = torch.arange(tgt_len, device=device).unsqueeze(0)

    src_emb = self.tok_embed(src) + self.pos_embed(src_positions)
    tgt_emb = self.tok_embed(tgt) + self.pos_embed(tgt_positions)

    enc = src_emb
    for layer in self.enc_layers:
      enc = layer(enc)

    dec = tgt_emb
    for layer in self.dec_layers:
      dec = layer(dec, enc, mask=tgt_mask)

    return self.fc_out(dec)
"""


'\nclass Transformer(nn.Module):\n  def __init__(self, vocab_size, embed_dim, heads, ff_dim, depth, max_len=128):\n    super().__init__()\n    \n    self.tok_embed = nn.Embedding(vocab_size, embed_dim)\n    self.pos_embed = nn.Embedding(max_len, embed_dim)\n\n    self.enc_layers = nn.ModuleList([\n        EncoderLayer(embed_dim, heads, ff_dim) for _ in range(depth)\n    ])\n\n    self.dec_layers = nn.ModuleList([\n        DecoderLayer(embed_dim, heads, ff_dim) for _ in range(depth)\n    ])\n\n    self.fc_out = nn.Linear(embed_dim, vocab_size)\n\n  def forward(self, src, tgt, tgt_mask=None):\n    batch, src_len = src.shape\n    _, tgt_len = tgt.shape\n    device = src.device\n\n    src_positions = torch.arange(src_len, device=device).unsqueeze(0)\n    tgt_positions = torch.arange(tgt_len, device=device).unsqueeze(0)\n\n    src_emb = self.tok_embed(src) + self.pos_embed(src_positions)\n    tgt_emb = self.tok_embed(tgt) + self.pos_embed(tgt_positions)\n\n    enc = src_emb\n    for layer i

In [11]:
# Mask generation & model implementation
def generate_square_subsequent_mask(seq_len):
  """
  디코더용 마스크 생성 함수.
  현재 시점 이후의 위치를 True로 표시하여 어텐션 계산에서 제외
  반환 shape: (1, 1, seq_len, seq_len) -> (batch, heads, seq_q, seq_k)에 브로드캐스트 가능
  """
  # Upper triangle을 1로 채운 뒤 bool로 변환
  mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
  # batch, head dim 추가
  return mask.unsqueeze(0).unsqueeze(0)

# 간단한 동작 테스트
batch = 2
src = torch.randint(0, 50, (batch,6))  # batch, src_len
tgt = torch.randint(0, 50, (batch,6))  # batch, tgt_len
tgt_mask = generate_square_subsequent_mask(tgt.size(1))

model = Transformer(
    vocab_size=50,
    embed_dim=32,
    heads=4,
    ff_dim=64,
    depth=2,
    max_len=64
)

out = model(src, tgt, tgt_mask=tgt_mask)
print(out.shape)   # (batch, tgt_len, vocab_size)

torch.Size([2, 6, 50])
